# Classificação de Grãos de Café com EfficientNetB0 e Fine-Tuning

Projeto final da disciplina de Processamento de Imagens.

Dataset: Coffee Bean Dataset 224x224.

Classes: Dark, Green, Light e Medium.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/AlisonCarv/classificacao-cafe-efficientnet.git"
REPO_DIR = "/content/classificacao-cafe-efficientnet"

if os.path.exists(REPO_DIR):
    print("Repositório já existe no Colab. Atualizando...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print("Clonando repositório...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Diretório atual:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!mkdir -p outputs/checkpoints
!mkdir -p outputs/modelos
!mkdir -p outputs/logs
!mkdir -p outputs/figuras
!mkdir -p outputs/metricas

In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/Kaggle Coffee Bean Dataset"
!ls "/content/drive/MyDrive/Colab Notebooks/Kaggle Coffee Bean Dataset/train"
!ls "/content/drive/MyDrive/Colab Notebooks/Kaggle Coffee Bean Dataset/test"

In [ ]:
from src import config

from src.dados import (
    exibir_resumo_csv,
    carregar_datasets
)

from src.modelo import (
    criar_modelo,
    preparar_fine_tuning
)

from src.treino import (
    criar_callbacks,
    compilar_modelo,
    treinar_fase_1,
    treinar_fase_2,
    salvar_modelo
)

from src.visualizacao import (
    plotar_amostras_dataset,
    plotar_historico
)

from src.avaliacao import avaliar_modelo

In [ ]:
df_dataset = exibir_resumo_csv()

In [ ]:
train_ds, val_ds, test_ds, class_names = carregar_datasets()

num_classes = len(class_names)

print("Classes:", class_names)
print("Número de classes:", num_classes)

In [ ]:
plotar_amostras_dataset(train_ds, class_names)

In [ ]:
modelo, modelo_base = criar_modelo(num_classes)
modelo.summary()

In [ ]:
callbacks = criar_callbacks()

In [ ]:
modelo = compilar_modelo(
    modelo,
    learning_rate=config.LR_FASE_1
)

modelo, historico_fase_1 = treinar_fase_1(
    modelo,
    train_ds,
    val_ds,
    callbacks
)

In [ ]:
plotar_historico(historico_fase_1, "fase_1")

In [ ]:
preparar_fine_tuning(
    modelo_base,
    num_camadas_liberadas=config.NUM_CAMADAS_FINE_TUNING
)

In [ ]:
modelo = compilar_modelo(
    modelo,
    learning_rate=config.LR_FASE_2
)

modelo, historico_fase_2 = treinar_fase_2(
    modelo,
    train_ds,
    val_ds,
    callbacks
)

In [ ]:
plotar_historico(historico_fase_2, "fase_2")

In [ ]:
salvar_modelo(modelo)

In [ ]:
acuracia_final = avaliar_modelo(
    modelo,
    test_ds,
    class_names
)

print(f"Acurácia final no teste: {acuracia_final:.4f}")